# Cypher Basics

## Overview
**Cypher** is Neo4j's declarative query language. Its defining feature is ASCII-art pattern syntax that mirrors how we draw graphs on whiteboards.

**Core pattern syntax:**
```cypher
(r:Researcher)-[:LEADS]->(t:Trial)
```
This reads: "a Researcher node connected by a LEADS relationship to a Trial node."

**The 6 most important Cypher clauses:**
```cypher
MATCH  (r:Researcher)-[:LEADS]->(t:Trial)   -- find the pattern
WHERE  t.phase >= 2                           -- filter
RETURN r.name, t.title                        -- return fields

CREATE (r:Researcher {id: 'R001', name: 'Aroha'})  -- create
MERGE  (r:Researcher {id: 'R001'})           -- upsert
SET    r.role = 'PI'                          -- update property
```

**SQLite/networkx note:** Cypher requires Neo4j. All demos in this folder use Python networkx for live results; full Cypher equivalents are shown alongside every query.

---

In [ ]:
print("=== Cypher fundamentals: ASCII-art pattern matching ===")
print()

cypher_patterns = [
    ("Node",                  "(n)",                    "Any node, bound to variable n"),
    ("Labelled node",         "(:Researcher)",          "Node with Researcher label"),
    ("Named + labelled",      "(r:Researcher)",         "Researcher node, variable r"),
    ("With property",         "({name: 'Aroha'})",      "Node where name = 'Aroha'"),
    ("Relationship",          "-[rel]->",               "Directed relationship, variable rel"),
    ("Typed relationship",    "-[:LEADS]->",            "Relationship of type LEADS"),
    ("With property",         "-[:TESTS {arm:'treatment'}]->","Relationship with property"),
    ("Undirected",            "-[:KNOWS]-",             "Undirected relationship"),
    ("Variable length",       "-[:KNOWS*1..3]-",        "1 to 3 hops"),
    ("Full pattern",          "(r:Researcher)-[:LEADS]->(t:Trial)", "Researcher LEADS Trial"),
]
print(f"  {'Element':<26s}  {'Cypher':<42s}  Meaning")
print("  " + "-"*90)
for element, cypher, meaning in cypher_patterns:
    print(f"  {element:<26s}  {cypher:<42s}  {meaning}")

print()
print("Core Cypher clauses:")
clauses = [
    ("MATCH",   "Find nodes/relationships matching a pattern"),
    ("CREATE",  "Create nodes and/or relationships"),
    ("MERGE",   "MATCH or CREATE if not found (upsert)"),
    ("WHERE",   "Filter matched results"),
    ("RETURN",  "Specify what to return"),
    ("SET",     "Set or update properties"),
    ("DELETE",  "Delete nodes or relationships"),
    ("DETACH DELETE", "Delete node and all its relationships"),
    ("WITH",    "Pipe results between query stages (like SQL CTE)"),
    ("UNWIND",  "Expand a list into rows (like SQL UNNEST)"),
    ("ORDER BY","Sort results"),
    ("LIMIT",   "Cap number of results"),
    ("SKIP",    "Pagination offset"),
]
for clause, desc in clauses:
    print(f"  {clause:<16s}  {desc}")


---
## CREATE, MERGE, SET, DELETE

In [ ]:
import networkx as nx

# Simulate Cypher CREATE / MATCH / MERGE with networkx
G = nx.MultiDiGraph()

print("=== CREATE: creating nodes and relationships ===")
create_examples = """
-- Create a Researcher node
CREATE (r:Researcher {id: 'R001', name: 'Aroha Ngata', role: 'PI'})

-- Create a Trial node
CREATE (t:Trial {id: 'T001', title: 'Cardio Alpha', phase: 2, status: 'active'})

-- Create both nodes + relationship in one statement
CREATE (r:Researcher {id: 'R001', name: 'Aroha Ngata'})
      -[:LEADS {since: '2023-01-01'}]->
       (t:Trial {id: 'T001', title: 'Cardio Alpha'})

-- Create relationship between existing nodes
MATCH (r:Researcher {id: 'R001'}), (t:Trial {id: 'T001'})
CREATE (r)-[:LEADS {role: 'PI'}]->(t)
"""
print(create_examples)

print("=== MERGE: upsert — match or create ===")
merge_examples = """
-- MERGE on a node (creates if not found; matches if exists)
MERGE (r:Researcher {id: 'R001'})
ON CREATE SET r.name = 'Aroha Ngata', r.created_at = datetime()
ON MATCH  SET r.last_seen = datetime()

-- MERGE on a relationship (safe idempotent writes)
MATCH (r:Researcher {id: 'R001'}), (t:Trial {id: 'T001'})
MERGE (r)-[:LEADS]->(t)

-- MERGE is always preferred over CREATE for idempotent pipelines
-- CREATE will duplicate nodes/relationships if run twice
-- MERGE will find the existing one and optionally update it
"""
print(merge_examples)

print("=== SET, REMOVE, DELETE ===")
dml_examples = """
-- Update a property
MATCH (r:Researcher {id: 'R001'})
SET   r.h_index = 15, r.updated_at = datetime()

-- Remove a property
MATCH (r:Researcher {id: 'R001'})
REMOVE r.temporary_flag

-- Add a label to an existing node
MATCH (r:Researcher {id: 'R001'})
SET   r:SeniorResearcher   -- node now has both :Researcher and :SeniorResearcher labels

-- Delete a relationship
MATCH (r:Researcher {id: 'R001'})-[rel:LEADS]->(t:Trial {id: 'T001'})
DELETE rel

-- Delete a node and ALL its relationships (safe)
MATCH (r:Researcher {id: 'R099'})
DETACH DELETE r
-- Plain DELETE fails if node still has relationships
"""
print(dml_examples)


---
## MATCH + WHERE: live pattern queries

In [ ]:
import networkx as nx

# Build the research graph
G = nx.MultiDiGraph()
nodes = [
    ("R001","Researcher",{"name":"Aroha Ngata","role":"PI"}),
    ("R002","Researcher",{"name":"Liam Chen","role":"Biostatistician"}),
    ("R003","Researcher",{"name":"Fatima Rashid","role":"Coordinator"}),
    ("R004","Researcher",{"name":"James MacLeod","role":"Researcher"}),
    ("I001","Institution",{"name":"Dalhousie University","country":"Canada"}),
    ("I002","Institution",{"name":"University of Toronto","country":"Canada"}),
    ("T001","Trial",{"title":"Cardio Alpha","phase":2}),
    ("T002","Trial",{"title":"Neuro Beta","phase":3}),
    ("D001","Disease",{"name":"Type 2 Diabetes","category":"Metabolic"}),
    ("D002","Disease",{"name":"Hypertension","category":"Cardiovascular"}),
    ("DR001","Drug",{"name":"Metformin","drug_class":"Biguanide"}),
    ("DR002","Drug",{"name":"Lisinopril","drug_class":"ACE Inhibitor"}),
    ("DR003","Drug",{"name":"Atorvastatin","drug_class":"Statin"}),
]
for nid, label, props in nodes:
    G.add_node(nid, label=label, **props)

edges = [
    ("R001","I001","AFFILIATED_WITH"),("R002","I002","AFFILIATED_WITH"),
    ("R003","I001","AFFILIATED_WITH"),("R004","I002","AFFILIATED_WITH"),
    ("R001","T001","LEADS"),("R004","T002","LEADS"),
    ("R002","T001","CONTRIBUTES_TO"),("R002","T002","CONTRIBUTES_TO"),
    ("R003","T001","CONTRIBUTES_TO"),
    ("T001","D002","STUDIES"),("T001","D001","STUDIES"),
    ("T002","D002","STUDIES"),
    ("T001","DR002","TESTS"),("T001","DR001","TESTS"),
    ("T002","DR003","TESTS"),
    ("DR001","D001","TREATS"),("DR002","D002","TREATS"),
    ("DR003","D002","TREATS"),
    ("R001","R004","COLLABORATES_WITH"),
]
for s, d, r in edges:
    G.add_edge(s, d, rel_type=r)

print("=== MATCH + WHERE: pattern queries ===")
print()

# Q1: All researchers
print("Q1: All Researcher nodes")
print("Cypher: MATCH (r:Researcher) RETURN r.name, r.role ORDER BY r.name")
researchers = [(d['name'], d['role']) for n, d in G.nodes(data=True) if d.get('label')=='Researcher']
for name, role in sorted(researchers):
    print(f"  {name:<20s}  {role}")

print()
# Q2: PI researchers
print("Q2: Researchers who are PIs")
print("Cypher: MATCH (r:Researcher) WHERE r.role = 'PI' RETURN r.name")
pis = [d['name'] for n, d in G.nodes(data=True) if d.get('label')=='Researcher' and d.get('role')=='PI']
for p in sorted(pis): print(f"  {p}")

print()
# Q3: Trials led by each researcher
print("Q3: Trials and the researchers who lead them")
print("Cypher: MATCH (r:Researcher)-[:LEADS]->(t:Trial) RETURN r.name, t.title, t.phase")
for u, v, d in G.edges(data=True):
    if d['rel_type'] == 'LEADS':
        rname = G.nodes[u]['name']
        ttitle = G.nodes[v]['title']
        tphase = G.nodes[v]['phase']
        print(f"  {rname:<20s}  leads  {ttitle}  (Phase {tphase})")

print()
# Q4: Drugs tested in Phase 2+ trials
print("Q4: Drugs tested in Phase 2+ trials")
print("Cypher: MATCH (t:Trial)-[:TESTS]->(d:Drug) WHERE t.phase >= 2 RETURN t.title, d.name")
for u, v, d in G.edges(data=True):
    if d['rel_type'] == 'TESTS':
        tphase = G.nodes[u].get('phase', 0)
        if tphase >= 2:
            print(f"  {G.nodes[u]['title']}  tests  {G.nodes[v]['name']}")


---
## Common Pitfalls

**1. Using CREATE instead of MERGE in ETL pipelines**
`CREATE` always creates a new node or relationship. Running an ETL job twice with `CREATE` doubles every node and edge. Always use `MERGE` for idempotent data loading.

**2. MERGE on multiple properties creates unexpected nodes**
`MERGE (r:Researcher {name: 'Aroha', role: 'PI'})` matches a node only if BOTH name AND role match exactly. If a node exists with the right name but a different role, MERGE creates a new node. MERGE should be on the unique identifier only: `MERGE (r:Researcher {id: $id}) SET r.name = $name, r.role = $role`.

**3. DETACH DELETE on large subgraphs timing out**
`MATCH (n) DETACH DELETE n` deletes all nodes and relationships but can time out or run out of memory on large graphs. Delete in batches: `MATCH (n) WITH n LIMIT 10000 DETACH DELETE n` and repeat.

**4. Forgetting WHERE r1 <> r2 in self-relationship queries**
`MATCH (r1:Researcher)-[*1..2]-(r2:Researcher)` without `WHERE r1 <> r2` returns each researcher as their own 0-hop result (a node can reach itself via zero hops). Always exclude the starting node from results.

**5. Not using parameters ($var) — query plan cache misses and injection risk**
`session.run(f"MATCH (r:Researcher {{name: '{name}'}}) RETURN r")` bypasses the query plan cache (different string each call) and is a Cypher injection risk. Always use parameters: `session.run('MATCH (r:Researcher {name: $name}) RETURN r', name=name)`.

**6. Unbounded variable-length paths crashing the database**
`MATCH p = (a)-[*]->(b)` with no depth limit on a well-connected graph explores every possible path — exponential combinations. Always set a maximum: `[*..5]`. Use `shortestPath()` when you only need the minimum-hop route.


---
*sql_methods_library - Samantha McGarrigle*